# 02 — Prepare the Dataset

**Pipeline:** Dataset → EDA → **Preprocessing → Augmentation** → Training → Evaluation → Error Analysis → Inference → Deployment

Three jobs, all driven by `configs/config.yaml`:
1. **Split** — stratified train/val/test so every split keeps the class balance, fixed by `seed`
2. **Preprocess** — resize + ImageNet normalization (required by the pretrained backbones)
3. **Augment** — train split only; val/test stay deterministic so metrics are honest

In [ ]:
# run from notebooks/ -- make the repo root importable and the cwd
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

from src.utils import load_config, seed_everything
cfg = load_config("configs/config.yaml")
seed_everything(cfg["seed"])
print("project:", cfg["project_name"], "| data root:", cfg["data"]["root"])

## The split, verified

Stratified per class — check the proportions survive.

In [ ]:
from src.dataset import build_datasets
import matplotlib.pyplot as plt
import numpy as np

ds = build_datasets(cfg)
classes = ds["classes"]
targets = [t for _, t in ds["samples"]]

fig, ax = plt.subplots(figsize=(9, 3.5))
width = 0.27
for k, (name, idxs) in enumerate(ds["split_indices"].items()):
    per = [sum(1 for i in idxs if targets[i] == c) for c in range(len(classes))]
    ax.bar(np.arange(len(classes)) + (k - 1) * width, per, width, label=f"{name} ({len(idxs)})")
ax.set_xticks(range(len(classes)), classes, rotation=30, ha="right")
ax.legend(); ax.set_title("class balance per split (stratified)")
plt.tight_layout(); plt.show()

## What augmentation actually does

Same image, eight draws of the train transform. If an augmented image could plausibly change class, the augmentation is too aggressive — that's why rotation defaults to a mild 15 degrees.

In [ ]:
from src.dataset import build_transforms, IMAGENET_MEAN, IMAGENET_STD
from PIL import Image
import torch

def denorm(x):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1).permute(1, 2, 0)

train_tf = build_transforms(cfg, train=True)
eval_tf = build_transforms(cfg, train=False)
path, target = ds["samples"][0]
img = Image.open(path).convert("RGB")

fig, axes = plt.subplots(1, 9, figsize=(18, 2.4))
axes[0].imshow(denorm(eval_tf(img))); axes[0].set_title("eval (deterministic)", fontsize=9)
for i in range(1, 9):
    axes[i].imshow(denorm(train_tf(img))); axes[i].set_title(f"train draw {i}", fontsize=9)
for ax in axes: ax.axis("off")
plt.suptitle(f"class: {ds['classes'][target]} -- augment config: {cfg['data']['augment']}")
plt.tight_layout(); plt.show()

## Batches, as the model will see them

In [ ]:
from src.dataset import build_loaders

loaders = build_loaders(cfg)
images, labels = next(iter(loaders["train"]))
print("batch tensor:", tuple(images.shape), images.dtype, f"| range [{images.min():.2f}, {images.max():.2f}]")

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for ax, img_t, lab in zip(axes.ravel(), images, labels):
    ax.imshow(denorm(img_t)); ax.set_title(loaders["classes"][lab], fontsize=8); ax.axis("off")
plt.suptitle("one training batch (denormalized for display)")
plt.tight_layout(); plt.show()

## Tuning notes

| Symptom | Try |
|---|---|
| Overfits fast (train acc much higher than val acc) | more augmentation, `random_erasing: 0.25`, higher dropout |
| Underfits / augmented images look wrong | reduce rotation / jitter; augmentation must preserve the label |
| Classes imbalanced | weighted loss or oversampling — the split keeps ratios, it doesn't fix them |

Next: **[03 — Train the Model](03_train_model.ipynb)**.